# Neural network lab ? Colab workflow

Run the sections in order after selecting a GPU runtime. Training, evaluation, installation, and Drive backup cells are opt-in. Replace the marked decisions after seeing real validation results. Keep outputs when saving the finished notebook. The validation split is fixed at 20% of the supplied train set with seed 42; eval is reserved for the final scoring step.


## Part 0 ? Setup and Data

**Optional repository setup.** If the repository is not already available in Colab, enter its public Git URL below and run the next cell. Leave `REPO_URL` empty when the files are already present. No credentials belong in this notebook.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = ""  # Optional public HTTPS Git URL; set explicitly to clone.
CLONE_DIR = Path("/content/K4-Track4-Day1-Neuralnetwork")
if REPO_URL:
    if CLONE_DIR.exists():
        print(f"Repository already exists: {CLONE_DIR}. Pull manually if needed.")
    else:
        subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)


The next cell accepts a repository `code/` directory or `submission_STUDENT_ID/code/`. If Colab opened this notebook from another working directory, set `CODE_DIR_OVERRIDE` to that directory. Results and figures go beside `code/`; the processed data and scripts come from the repository root.


In [ ]:
import sys
import importlib.util

CODE_DIR_OVERRIDE = None  # Example: Path("/content/my-repo/code")
candidates = ([Path(CODE_DIR_OVERRIDE)] if CODE_DIR_OVERRIDE else []) + [
    Path.cwd(), Path.cwd() / "code", CLONE_DIR / "code"
]
CODE_DIR = next((p.resolve() for p in candidates if (p / "train.py").is_file()), None)
if CODE_DIR is None:
    raise FileNotFoundError("Set CODE_DIR_OVERRIDE to the directory containing train.py")
if (CODE_DIR.parent / "scripts" / "split_data.py").is_file():
    REPO_ROOT = CODE_DIR.parent
elif (CODE_DIR.parent.parent / "scripts" / "split_data.py").is_file():
    REPO_ROOT = CODE_DIR.parent.parent
else:
    raise FileNotFoundError("Could not find repository scripts/split_data.py")
OUT_DIR = CODE_DIR.parent
RESULTS_DIR = OUT_DIR / "results"
FIGURES_DIR = OUT_DIR / "figures"
sys.path.insert(0, str(CODE_DIR))
LIGHTWEIGHT_PACKAGES = ("numpy", "scikit-learn", "matplotlib", "openpyxl")
PACKAGE_MODULES = {"numpy": "numpy", "scikit-learn": "sklearn",
                   "matplotlib": "matplotlib", "openpyxl": "openpyxl"}
missing = [name for name in LIGHTWEIGHT_PACKAGES
           if importlib.util.find_spec(PACKAGE_MODULES[name]) is None]
print("Code:", CODE_DIR, "Repo:", REPO_ROOT, "Output:", OUT_DIR)
print("Missing optional/required lightweight packages:", missing or "none")
print("PyTorch available:", importlib.util.find_spec("torch") is not None)


Optional dependency installation: change the flag to `True` only if you choose to install missing lightweight packages. This cell never reinstalls PyTorch or changes project requirements.


In [ ]:
INSTALL_MISSING_LIGHTWEIGHT = False
if INSTALL_MISSING_LIGHTWEIGHT and missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)


In [ ]:
import json
import math
import shutil
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, activation_stats
from train import DEFAULT_CFG, set_seed, evaluate, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)


Create the fixed processed train/eval files only if either is absent. This uses the supplied metadata and does not make a new train/eval split. `prepare_data` then makes the stratified validation split and fits the standardizer on the remaining train portion.


In [ ]:
processed_dir = REPO_ROOT / "data" / "processed"
if not all((processed_dir / name).is_file() for name in ("train.npz", "eval.npz")):
    subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "split_data.py")],
                   cwd=REPO_ROOT, check=True)
data = prepare_data(str(device), val_fraction=0.2, seed=42,
                    processed_dir=str(processed_dir))


In [ ]:
for split in ("tr", "val", "eval"):
    X, y = data[f"X_{split}"], data[f"y_{split}"]
    counts = torch.bincount(y, minlength=7).cpu().numpy()
    print(f"{split}: X={tuple(X.shape)}, y={tuple(y.shape)}, class counts={counts.tolist()}")
    print("  class proportions:", np.round(counts / counts.sum(), 4).tolist())
majority = torch.bincount(data["y_tr"], minlength=7).argmax()
majority_acc = (data["y_val"] == majority).float().mean().item()
print(f"Validation majority-class accuracy: {majority_acc:.4f}")
numeric = data["X_tr"][:, :10]
print("Train numeric means:", numeric.mean(0).cpu().numpy().round(3).tolist())
print("Train numeric stds:", numeric.std(0, unbiased=False).cpu().numpy().round(3).tolist())
# Eval class counts above are descriptive only; never use eval labels to select a model.


### Optional Google Drive backup

Leave this section disabled unless you want Drive backup. Set `DRIVE_BACKUP_DIR` to your own folder under `/content/drive`; no personal path is embedded here. Save the notebook in Colab or Drive to retain its cell outputs.


In [ ]:
MOUNT_DRIVE = False
BACKUP_TO_DRIVE = False
DRIVE_BACKUP_DIR = None  # Example: Path("/content/drive/MyDrive/<your-folder>")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if BACKUP_TO_DRIVE:
    if DRIVE_BACKUP_DIR is None:
        raise ValueError("Set DRIVE_BACKUP_DIR before backing up")
    backup_dir = Path(DRIVE_BACKUP_DIR).expanduser()
    backup_dir.mkdir(parents=True, exist_ok=True)
    for source in (RESULTS_DIR, FIGURES_DIR):
        if source.exists():
            shutil.copytree(source, backup_dir / source.name, dirs_exist_ok=True)
    print("Backed up results and figures to", backup_dir)


## Part 1 ? Model Sanity Checks

M-base has 47,879 trainable parameters and returns raw logits for seven classes. These checks use one random batch; they do not train the baseline.


In [ ]:
set_seed(1)
sanity_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(sanity_model) == EXPECTED_PARAMS[(256, 128)] == 47_879
random_x = torch.randn(8, 54, device=device)
random_y = torch.randint(0, 7, (8,), device=device)
logits = sanity_model(random_x)
assert logits.shape == (8, 7) and torch.isfinite(logits).all()
F.cross_entropy(logits, random_y).backward()
for name, parameter in sanity_model.named_parameters():
    assert parameter.grad is not None, name
    print(name, "gradient L2 norm:", parameter.grad.norm().item())
print("Parameter count:", count_params(sanity_model), "logit shape:", tuple(logits.shape))


In [ ]:
step0_ce = evaluate(sanity_model, data["X_val"], data["y_val"], "ce")["loss"]
print(f"ln(7): {math.log(7):.6f}; measured step-0 validation CE: {step0_ce:.6f}")
assert math.isfinite(step0_ce)


**Step-0 observation (fill after running):** Is the measured loss finite and reasonably close to `ln(7)`? Record the seed and explain any noticeable difference. Random initialization does not guarantee an exact value of 1.946.


In [ ]:
# Isolated 20-sample diagnostic; no validation or eval samples enter this training.
set_seed(1)
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
indices = torch.randperm(len(data["X_tr"]), generator=torch.Generator().manual_seed(42))[:20].to(device)
small_x, small_y = data["X_tr"][indices], data["y_tr"][indices]
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
overfit_losses = []
for step in range(300):
    overfit_model.train()
    overfit_optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(overfit_model(small_x), small_y)
    loss.backward()
    overfit_optimizer.step()
    overfit_model.eval()
    with torch.no_grad():
        current_logits = overfit_model(small_x)
        current_loss = F.cross_entropy(current_logits, small_y).item()
        current_acc = (current_logits.argmax(1) == small_y).float().mean().item()
    overfit_losses.append(current_loss)
    if step % 25 == 0 or (current_acc == 1.0 and current_loss < 1e-4):
        print(f"step {step + 1}: loss={current_loss:.6g}, acc={current_acc:.3f}")
    if current_acc == 1.0 and current_loss < 1e-4:
        break
plt.plot(range(1, len(overfit_losses) + 1), overfit_losses)
plt.xlabel("Update step"); plt.ylabel("CE loss on 20 samples")
plt.title("Small-batch overfit diagnostic"); plt.show()


**Small-batch observation (fill after running):** Final loss: ___. Final accuracy: ___. Did loss decrease toward zero and accuracy reach or approach 100%? If not, inspect labels, logits, gradients, and the update step before longer training.


## Part 2 ? Baseline

`DEFAULT_CFG` fixes M-base `(256,128)`, He, CE, SGD momentum 0.9, batch 512, 20 epochs, zero dropout, no clipping, and FP32. Its learning rate stays unset until selected from validation. `run_and_save` calls the shared training pipeline and saves JSON and PNG; `best_state` remains in memory because JSON intentionally excludes weights.


In [ ]:
RUN_RESULTS = {}
def run_and_save(cfg, data):
    if cfg["lr"] is None:
        raise ValueError("Choose cfg['lr'] using validation before running")
    result = run_experiment(cfg, data)
    save_result(result, str(RESULTS_DIR))
    plot_run(result, str(FIGURES_DIR / f"{cfg['exp_id']}.png"))
    RUN_RESULTS[cfg["exp_id"]] = result
    print(cfg["exp_id"], result["summary"])
    return result


### Baseline learning-rate search

The three candidates share the same 5-epoch pilot budget and seed. Enable the cell to run them. Inspect their validation curves and metrics, then explicitly choose the official baseline LR below. Pilot results are separate from 20-epoch official baseline runs.


In [ ]:
RUN_LR_SEARCH = False
LR_SEARCH_EPOCHS = 5
lr_search_cfgs = [
    {**DEFAULT_CFG, "exp_id": f"lr-sgdm-{lr:g}", "group": "lr_search",
     "description": f"SGD momentum LR pilot {lr:g}", "lr": lr,
     "epochs": LR_SEARCH_EPOCHS, "seed": 1}
    for lr in (0.01, 0.03, 0.1)
]
if RUN_LR_SEARCH:
    lr_search_results = [run_and_save(candidate, data) for candidate in lr_search_cfgs]
    for result in lr_search_results:
        summary = result["summary"]
        print(result["cfg"]["lr"], summary["best_val_loss"],
              summary["val_macro_f1"], summary["diverged"])


**LR decision (fill after pilots):** Set `BASELINE_LR` using validation loss, macro-F1, stability, and curves. Record why it was chosen. Do not consult eval.


In [ ]:
BASELINE_LR = None  # Replace with the validation-chosen LR, e.g. one tested above.
base_cfg = {**DEFAULT_CFG, "lr": BASELINE_LR}


### Official baseline seeds

Enable after setting `BASELINE_LR`. Seeds 1 and 2 are prepared; add seed 3 if useful. Each run is 20 epochs and gets its own JSON and three-panel plot.


In [ ]:
RUN_BASELINE_SEEDS = False
BASELINE_SEEDS = (1, 2)  # Optionally use (1, 2, 3).
if RUN_BASELINE_SEEDS:
    if BASELINE_LR is None:
        raise ValueError("Set BASELINE_LR from validation before official baseline runs")
    baseline_runs = [
        run_and_save({**base_cfg, "seed": seed, "exp_id": f"base-s{seed}",
                      "description": f"Baseline M-base, seed {seed}"}, data)
        for seed in BASELINE_SEEDS
    ]


In [ ]:
baseline_saved = [r for r in load_results(str(RESULTS_DIR))
                  if r["cfg"]["group"] == "baseline"]
if baseline_saved:
    for key in ("val_macro_f1", "val_acc", "best_val_loss"):
        values = np.array([r["summary"][key] for r in baseline_saved], dtype=float)
        if len(values) > 1:
            print(key, "mean:", values.mean(), "std:", values.std(ddof=1))
        else:
            print(key, "one seed only; seed noise cannot be estimated")
else:
    print("No official baseline results saved yet")


**Baseline observation (fill after running):** Describe train/validation curves, best epoch, validation accuracy relative to the majority-class reference, and seed mean ? standard deviation. If only one seed ran, record that uncertainty.


## Part 3 ? Experiments

For each topic, write a prediction **before** enabling its execution cell. Keep the split, seed, epochs, and all unrelated settings fixed. Results below are configuration menus, not completed experiments. Set only the topic flag you want to run. Every executed configuration passes through `run_and_save`.


### 1. Loss

**Prediction before running:** ___

Compare CE with one-hot MSE using validation accuracy, macro-F1, and convergence. Raw CE and MSE losses have different scales and should not be compared directly.


In [ ]:
loss_cfgs = [
    {**base_cfg, "exp_id": "loss-ce", "group": "loss", "description": "CE reference", "loss": "ce"},
    {**base_cfg, "exp_id": "loss-mse", "group": "loss", "description": "One-hot MSE", "loss": "mse"},
]


In [ ]:
RUN_LOSS = False
if RUN_LOSS:
    loss_runs = [run_and_save(candidate, data) for candidate in loss_cfgs]


**Observation after running:** ___  Which validation metrics and curve shapes changed? Did this match the prediction?


### 2. Optimizer

**Prediction before running:** ___

Each optimizer receives three LR candidates. Weight decay remains zero so the primary changed factor is the optimizer; Adam and AdamW can coincide when weight decay is zero. Compare each optimizer after its own LR search on validation.


In [ ]:
optimizer_lrs = {
    "sgd_momentum": (0.01, 0.03, 0.1),
    "adam": (3e-4, 1e-3, 3e-3),
    "adamw": (3e-4, 1e-3, 3e-3),
}
optimizer_cfgs = [
    {**base_cfg, "exp_id": f"opt-{name}-{lr:g}", "group": "optimizer",
     "description": f"{name} LR {lr:g}", "optimizer": name, "lr": lr,
     "weight_decay": 0.0}
    for name, candidates in optimizer_lrs.items() for lr in candidates
]


In [ ]:
RUN_OPTIMIZER = False
if RUN_OPTIMIZER:
    optimizer_runs = [run_and_save(candidate, data) for candidate in optimizer_cfgs]


In [ ]:
from collections import defaultdict
optimizer_saved = [r for r in load_results(str(RESULTS_DIR))
                   if r["cfg"]["group"] == "optimizer" and not r["summary"]["diverged"]]
by_optimizer = defaultdict(list)
for result in optimizer_saved:
    by_optimizer[result["cfg"]["optimizer"]].append(result)
for name, candidates in sorted(by_optimizer.items()):
    best = max(candidates, key=lambda r: r["summary"]["val_macro_f1"])
    print(name, "best tested validation candidate:", best["cfg"]["exp_id"],
          "lr:", best["cfg"]["lr"], "val macro-F1:", best["summary"]["val_macro_f1"])


**Observation after running:** ___  Compare each optimizer at its best tested validation LR, then consider seed noise and curve stability before drawing a conclusion.


### 3. Hyperparameter ? batch size

**Prediction before running:** ___

Change batch size only: 128, 512, 2048. Same epoch count gives different numbers of optimizer updates; account for that and epoch time in the interpretation.


In [ ]:
hparam_cfgs = [
    {**base_cfg, "exp_id": f"batch-{batch}", "group": "hparam",
     "description": f"Batch size {batch}", "batch": batch}
    for batch in (128, 512, 2048)
]


In [ ]:
RUN_HPARAM = False
if RUN_HPARAM:
    hparam_runs = [run_and_save(candidate, data) for candidate in hparam_cfgs]


**Observation after running:** ___  Include validation metrics, update counts, and time per epoch.


### 4. Dropout

**Prediction before running:** ___

Test 0.0, 0.3, 0.5 after inspecting baseline train/validation curves. Dropout is most meaningful when baseline shows overfitting.


In [ ]:
dropout_cfgs = [
    {**base_cfg, "exp_id": f"drop-{rate:g}", "group": "dropout",
     "description": f"Dropout {rate:g}", "dropout": rate}
    for rate in (0.0, 0.3, 0.5)
]


In [ ]:
RUN_DROPOUT = False
if RUN_DROPOUT:
    dropout_runs = [run_and_save(candidate, data) for candidate in dropout_cfgs]


**Observation after running:** ___  Did the train/validation gap indicate overfitting, and did dropout change validation metrics?


### 5. Gradient clipping

**Prediction before running:** ___

First inspect the baseline's recorded average global gradient norm before clipping. Choose a high LR and `CLIP_NORM` from observed gradients; no threshold is presumed correct. The paired high-LR runs differ only in clipping.


In [ ]:
baseline_reference = next((r for r in load_results(str(RESULTS_DIR))
                           if r["cfg"]["exp_id"] == "base-s1"), None)
if baseline_reference is None:
    print("Run baseline seed 1 before choosing a clipping threshold")
else:
    print("Baseline average pre-clip grad norm by epoch:",
          baseline_reference["history"]["grad_norm"])


In [ ]:
HIGH_LR = None  # Choose from validation/pilot behavior.
CLIP_NORM = None  # Choose after inspecting baseline gradient norms.
clip_cfgs = []
if HIGH_LR is not None and CLIP_NORM is not None:
    clip_cfgs = [
        {**base_cfg, "exp_id": "clip-highlr-none", "group": "clipping",
         "description": "High LR without clipping", "lr": HIGH_LR, "clip_norm": None},
        {**base_cfg, "exp_id": f"clip-highlr-{CLIP_NORM:g}", "group": "clipping",
         "description": f"High LR with clip {CLIP_NORM:g}", "lr": HIGH_LR,
         "clip_norm": CLIP_NORM},
    ]


In [ ]:
RUN_CLIPPING = False
if RUN_CLIPPING:
    if not clip_cfgs:
        raise ValueError("Set HIGH_LR and CLIP_NORM from observed validation/gradients")
    clipping_runs = [run_and_save(candidate, data) for candidate in clip_cfgs]


**Observation after running:** ___  Was clipping active at this threshold? Compare divergence, gradients, and validation metrics for the matched high-LR pair.


### 6. Mixed precision

**Prediction before running:** ___

Compare FP32 and FP16 on CUDA; add BF16 only when `torch.cuda.is_bf16_supported()` returns true. Compare validation metrics, time per epoch, and peak allocated GPU memory. Precision is not expected to improve accuracy by itself.


In [ ]:
amp_modes = ["fp32"]
if device.type == "cuda":
    amp_modes.append("fp16")
    if torch.cuda.is_bf16_supported():
        amp_modes.append("bf16")
amp_cfgs = [
    {**base_cfg, "exp_id": f"amp-{precision}", "group": "amp",
     "description": f"Precision {precision}", "precision": precision}
    for precision in amp_modes
]
print("Supported comparison modes:", amp_modes)


In [ ]:
RUN_AMP = False
if RUN_AMP:
    if device.type != "cuda":
        raise RuntimeError("Select a CUDA runtime for the mixed-precision comparison")
    amp_runs = [run_and_save(candidate, data) for candidate in amp_cfgs]
    for result in amp_runs:
        s = result["summary"]
        print(result["cfg"]["precision"], "val macro-F1:", s["val_macro_f1"],
              "seconds/epoch:", s["time_per_epoch_s"], "peak MB:", s["peak_mem_MB"])


**Observation after running:** ___  Explain speed, memory, and stability differences, including a lack of speedup if observed.


### 7. Initialization

**Prediction before running:** ___

Compare He, Xavier, small normal, and PyTorch default with all other configuration fields fixed. Optionally enable zero initialization as a symmetry diagnostic, not as a competitive configuration. Inspect activation standard deviations on a fixed validation batch before training.


In [ ]:
INCLUDE_ZEROS_DIAGNOSTIC = False
init_modes = ["he", "xavier", "normal", "default"]
if INCLUDE_ZEROS_DIAGNOSTIC:
    init_modes.append("zeros")
init_cfgs = [
    {**base_cfg, "exp_id": f"init-{mode}", "group": "init",
     "description": f"Initialization {mode}", "init": mode}
    for mode in init_modes
]
fixed_val_x = data["X_val"][:512]
for mode in init_modes:
    set_seed(base_cfg["seed"])
    probe = MLP(hidden=base_cfg["hidden"], dropout=0.0, init=mode).to(device)
    print(mode, "activation std after each Linear:", activation_stats(probe, fixed_val_x))


In [ ]:
RUN_INIT = False
if RUN_INIT:
    init_runs = [run_and_save(candidate, data) for candidate in init_cfgs]


**Observation after running:** ___  Relate measured activation scales and validation curves to the initialization mechanism. If zeros were run, discuss symmetry.


### Compare saved validation results

Saved JSON has configuration, history, and summary but no model weights. Filter by `group`, inspect validation scores and stability, and generate group comparison figures when requested. Do not use eval here.


In [ ]:
saved_results = load_results(str(RESULTS_DIR))
for group in sorted({r["cfg"]["group"] for r in saved_results}):
    print("GROUP", group)
    for result in (r for r in saved_results if r["cfg"]["group"] == group):
        cfg, s = result["cfg"], result["summary"]
        print(cfg["exp_id"], "val F1:", s["val_macro_f1"],
              "val loss:", s["best_val_loss"], "diverged:", s["diverged"])


In [ ]:
RUN_COMPARISON_PLOTS = False
COMPARE_GROUP = "optimizer"
COMPARE_METRIC = "val_macro_f1"
if RUN_COMPARISON_PLOTS:
    selected = [r for r in load_results(str(RESULTS_DIR))
                if r["cfg"]["group"] == COMPARE_GROUP]
    if len(selected) < 2:
        raise ValueError(f"Need at least two saved results in group {COMPARE_GROUP!r}")
    plot_compare(selected, COMPARE_METRIC,
                 str(FIGURES_DIR / f"compare_{COMPARE_GROUP}.png"),
                 title=f"{COMPARE_GROUP}: {COMPARE_METRIC}")


## Part 4 ? Final Selection and Evaluation

### Select final configuration using validation only

Review validation macro-F1, validation loss, baseline seed noise, divergence, and training cost. The table below uses saved validation results only. Explicitly define `FINAL_CFG` after review; no code selects a winner automatically.


In [ ]:
for result in load_results(str(RESULTS_DIR)):
    cfg, s = result["cfg"], result["summary"]
    print(cfg["exp_id"], "val macro-F1:", s["val_macro_f1"],
          "best val loss:", s["best_val_loss"],
          "seconds/epoch:", s["time_per_epoch_s"],
          "peak MB:", s["peak_mem_MB"], "diverged:", s["diverged"])


In [ ]:
FINAL_CFG = None  # Explicitly set to a complete chosen configuration from validation.
# Example after choosing: FINAL_CFG = dict(chosen_cfg)
# Keep FINAL_CFG['exp_id'] equal to the chosen run's exp_id.


`save_result` does not store `best_state`. If the chosen run is still in `RUN_RESULTS`, use its in-memory best weights. After a Colab restart, explicitly set `RERUN_FINAL_IF_NEEDED=True` to rerun that configuration before prediction. Only enable final evaluation after the validation decision is recorded.


In [ ]:
RUN_FINAL_EVAL = False
RERUN_FINAL_IF_NEEDED = False
PREDICTIONS_PATH = OUT_DIR / "predictions_eval.csv"
EVAL_RESULT_PATH = OUT_DIR / "eval_result.json"
if RUN_FINAL_EVAL:
    if FINAL_CFG is None:
        raise ValueError("Define FINAL_CFG using validation before final evaluation")
    chosen_result = RUN_RESULTS.get(FINAL_CFG["exp_id"])
    if chosen_result is None and RERUN_FINAL_IF_NEEDED:
        chosen_result = run_and_save(FINAL_CFG, data)
    if chosen_result is None:
        raise RuntimeError("Best weights are not in memory; rerun explicitly")
    if chosen_result["cfg"] != FINAL_CFG:
        raise ValueError("FINAL_CFG differs from the in-memory trained configuration")
    final_eval(FINAL_CFG, chosen_result, data, str(PREDICTIONS_PATH))
    print("Predictions written:", PREDICTIONS_PATH)


### Separate final scoring command

Run only after `predictions_eval.csv` exists. The shell equivalent is `python scripts/evaluate.py --pred <predictions_eval.csv> --out <eval_result.json>`. This score is for reporting; **after viewing eval results, do not retune the configuration using eval**.


In [ ]:
RUN_EVAL_SCORER = False
if RUN_EVAL_SCORER:
    if not PREDICTIONS_PATH.is_file():
        raise FileNotFoundError(PREDICTIONS_PATH)
    subprocess.run([sys.executable, str(REPO_ROOT / "scripts" / "evaluate.py"),
                    "--pred", str(PREDICTIONS_PATH),
                    "--out", str(EVAL_RESULT_PATH)],
                   cwd=REPO_ROOT, check=True)


**Final eval observation (fill only after scoring):** Record accuracy, macro-F1, per-class F1, and confusion-matrix patterns from the real `eval_result.json`. Identify the hardest class and a likely source of confusion. No new configuration selection is allowed here.


### Create `experiments.xlsx` from real saved results

The template's `Legend`, `Experiments`, `Seeds`, and `Summary` sheets remain intact. `write_xlsx` fills the `Experiments` sheet by header and preserves formula cells. Only attach eval scores from an actual scorer JSON, and only to the matching final run. If you also score the baseline, add its real score separately. Fill `Seeds` and write short `Summary` observations in Excel after the workbook is created.


In [ ]:
RUN_XLSX = False
BASELINE_EVAL_JSON = None  # Optional path to a real baseline scorer JSON.
if RUN_XLSX:
    if importlib.util.find_spec("openpyxl") is None:
        raise ImportError("openpyxl is required to write experiments.xlsx")
    scored = {}
    if EVAL_RESULT_PATH.is_file():
        if FINAL_CFG is None:
            raise ValueError("Set FINAL_CFG before attributing final eval scores")
        scored[FINAL_CFG["exp_id"]] = json.loads(EVAL_RESULT_PATH.read_text(encoding="utf-8"))
    if BASELINE_EVAL_JSON is not None:
        scored["base-s1"] = json.loads(Path(BASELINE_EVAL_JSON).read_text(encoding="utf-8"))
    saved = load_results(str(RESULTS_DIR))
    if not saved:
        raise ValueError("No saved results to add to experiments.xlsx")
    absent_figures = [r["cfg"]["exp_id"] for r in saved
                      if not (FIGURES_DIR / f"{r['cfg']['exp_id']}.png").is_file()]
    if absent_figures:
        raise FileNotFoundError(f"Missing run figures: {absent_figures}")
    rows = [to_row(r, eval_scores=scored.get(r["cfg"]["exp_id"])) for r in saved]
    template = REPO_ROOT / "templates" / "experiment_table_template.xlsx"
    table_path = OUT_DIR / "experiments.xlsx"
    write_xlsx(rows, str(template), str(table_path))
    print("Workbook written:", table_path)


### Report and submission checklist

Set `STUDENT_ID` below; do not guess it. Prepare `submission_STUDENT_ID/` with `REPORT.md`, `experiments.xlsx`, `predictions_eval.csv`, `eval_result.json`, `figures/`, `results/`, and `code/` (including this notebook and the six modules). When running from `submission_STUDENT_ID/code/`, outputs already go to the submission folder. When running from repository `code/`, copy the finished outputs and code into the submission folder yourself; this notebook does not create it.

Start `REPORT.md` from `templates/REPORT_TEMPLATE.md`. Fill in the setup and data split, model and step-0/small-batch checks, baseline seed noise, and **only** the experiment topics actually run. Each topic needs its pre-run prediction, measured values linked to `exp_id` and figure, mechanism, and comparison against seed noise. Explain final selection from validation; then report real eval metrics, per-class F1, and confusion-matrix errors. Answer: **If loss does not decrease, what should be checked first?** Include limitations and unexpected outcomes. Every numerical claim must trace to a saved result or scorer output. Keep notebook outputs when saving.


In [ ]:
STUDENT_ID = "STUDENT_ID"  # Replace with your actual ID before submission.
SUBMISSION_DIR = REPO_ROOT / f"submission_{STUDENT_ID}"
print("Expected submission directory:", SUBMISSION_DIR)
print("Current output directory:", OUT_DIR)
